In [ ]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 100
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 2

2025-02-27 10:01:21.743381: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.4 which is older than the ptxas CUDA version (12.8.61). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
/home/ruanjohn/miniconda3/envs/mava/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas *= memory_config.decay_scaling_factor
decay_kappas = decay_kappas[None, :, None, None]

In [4]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=memory_config.decay_scaling_factor,
)

In [5]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros((bsz, retnet_num_heads, retnet_embed_dim//retnet_num_heads, retnet_embed_dim//retnet_num_heads))
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [6]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
)

In [7]:
hstate = copy.deepcopy(init_hstate)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):

    # todo: reset later
    hstate = hstate * decay_kappas
    obs_i = obs[:, step*num_agents:(step+1)*num_agents, ...]
    dones_i = dones[:, step*num_agents:(step+1)*num_agents]
    step_counts_i = step_counts[:, step*num_agents:(step+1)*num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, step_counts_i, method="recurrent")
    act_output.append(out)

In [8]:
act_output = jnp.concatenate(act_output, axis=1)

In [9]:
act_output.shape

(16, 400, 16)

In [10]:
hstate = copy.deepcopy(init_hstate)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts)

In [11]:
train_out.shape

(16, 400, 16)

In [12]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(0.04794142, dtype=float32)

In [13]:
jnp.abs(train_out - act_output)

Array([[[2.08519064e-02, 9.87196155e-03, 2.27766559e-02, ...,
         6.50418252e-02, 9.14803892e-02, 2.99853999e-02],
        [2.61178073e-02, 4.51044738e-02, 1.92723498e-02, ...,
         4.59878407e-02, 9.08255428e-02, 3.08712330e-02],
        [9.96754393e-02, 2.89864279e-02, 7.47429281e-02, ...,
         3.53455953e-02, 4.79825139e-02, 4.16145809e-02],
        ...,
        [6.06473535e-04, 5.32572810e-03, 1.64743774e-02, ...,
         2.08859518e-02, 2.55138800e-03, 2.68004797e-02],
        [6.20939136e-02, 4.08803299e-03, 3.65416110e-02, ...,
         2.68649384e-02, 7.69915059e-02, 1.04508020e-01],
        [4.84994501e-02, 1.43803610e-02, 6.68233335e-02, ...,
         4.08888571e-02, 1.45302892e-01, 7.55180568e-02]],

       [[5.43066338e-02, 5.73212840e-03, 4.35425015e-03, ...,
         5.03875501e-03, 1.36202713e-02, 8.63027759e-03],
        [5.13981283e-03, 2.46736389e-02, 1.76597089e-02, ...,
         2.42741182e-02, 2.51131020e-02, 1.98280439e-02],
        [8.08179099e-03, 